In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/leoknaw@gmail.com/consolidated_pipeline/setup/utilities

In [0]:
print(bronze)

In [0]:
# parameter widgets

dbutils.widgets.text("catalog", "haha", "Catalog")
dbutils.widgets.text("data_source", "aws", "Data Source")

In [0]:
data_source = dbutils.widgets.get("data_source")
catalog = dbutils.widgets.get("catalog")

base_path = f's3://atlikon/full_load/{data_source}/*.csv'

base_path

In [0]:
df = (
    spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(base_path) \
    .withColumn("read_timestamp", F.current_timestamp()) \
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

display(df)

In [0]:
df.write.format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f'{catalog}.{bronze}.{data_source}')

### Silver processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze}.{data_source}")
display(df_bronze)


In [0]:
duplicate_rows = df_bronze.groupBy(df_bronze.columns).count().filter("count > 1")
display(duplicate_rows)

In [0]:
df_silver = df_bronze.dropDuplicates(["customer_id"])
display(df_silver)

In [0]:
from pyspark.sql.functions import col

df_leading_spaces = df_bronze.filter(col("customer_name").rlike(r'^\s+'))
display(df_leading_spaces)

In [0]:
df_silver = df_silver.withColumn("customer_name", F.trim(df_silver["customer_name"]))

display(df_silver)


In [0]:
distinct_cities = df_silver.select("city").distinct()
display(distinct_cities)

In [0]:
# Example mapping of incorrect city names to correct ones
city_mapping = {
    "Bengalore": "Bengaluru",
    "Bengaluruu": "Bengalore",

    "NewDelhee": "New Delhi",
    "NewDheli": "New Delhi",
    "NewDelhi": "New Delhi",

    "Hyderbad": "Hyderabad",
    "Hyderabadd": "Hyderabad",
   
}

allowed = ["Bengaluru", "New Delhi", "Hyderabad"]

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType



df_silver = df_silver \
    .replace(city_mapping, subset=["city"]) \
    .withColumn("city", 
                F.when(F.col("city").isNull(), None) \
                 .when(F.col("city").isin(allowed), F.col("city")) \
                 .otherwise(None)
                
                )
    

display(df_silver)


In [0]:
df_silver = df_silver.withColumn("customer_name", 
                                 F.when(F.col("customer_name").isNull(), None) \
                                  .otherwise(F.initcap(F.col("customer_name"))))


display(df_silver)

In [0]:
display(df_silver.select("customer_name").distinct())

In [0]:
display(df_silver.filter(F.col("city").isNull()))

In [0]:
display(df_silver.select(["customer_name", "customer_id", "city"]))

In [0]:
id_mapping = {789403: "New Delhi", 789420: "Bengaluru", 789520: "Hyderabad", 789521: "New Delhi", 789603: "Hyderabad"}

df_id_mapping = spark.createDataFrame(
    [(k, v) for k, v in id_mapping.items()],
    ["customer_id", "fixed_city"]
)
display(df_id_mapping)

In [0]:
df_silver = df_silver.join(df_id_mapping, on="customer_id", how="left") \
    .withColumn("city", F.coalesce("city", "fixed_city")).drop("fixed_city")
display(df_silver)

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
display(df_silver)
df_silver.printSchema()

In [0]:
df_silver = df_silver.withColumn("customer",  \
    F.concat_ws("-", F.col("customer_name"), F.coalesce(F.col("city"), F.lit("unknown")))
)\
    .withColumn("market", F.lit("India")) \
    .withColumn("platform", F.lit("Sports bar")) \
    .withColumn("channel", F.lit("Acquisition"))
    
    
display(df_silver)

        
        

In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver}.{data_source}")

### Gold processing

In [0]:
df_gold = df_silver.select(["customer_id", "customer_name","customer","city", "market", "platform", "channel"])
display(df_gold)


In [0]:
df_gold.write.mode("overwrite") \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold}.{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark, f"{catalog}.{gold}.dim_customers")
df_child_customers = spark.table(f"{catalog}.{gold}.customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()